# SAE Eval — pull finished runs from Weights & Biases

Reads the completed training runs from the `siglip-sae` wandb project,
extracts final per-layer metrics, writes them to `results/`, and plots them.
Runs locally (no GPU). Uses the metric keys logged by the ViT-Prisma trainer.


## 1 — Install + login
Paste the same wandb key you used in SAE_Training.ipynb.


In [ ]:
!pip install -q wandb pandas matplotlib
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize


## 2 — Config


In [ ]:
ENTITY  = None                 # your wandb username; None = default entity
PROJECT = 'siglip-sae'

# Which run family to pull. Runs are named like:
#   siglip-base-patch16-224_hook_resid_post_topk_32_layer{N}
MODEL_TAG  = 'siglip-base-patch16-224'
HOOK_POINT = 'hook_resid_post'
RUN_TAG    = 'topk_32'         # matches the TAG used in training

OUT_CSV = f'results/sae_metrics_{RUN_TAG}.csv'


## 3 — Pull final metrics from wandb


In [ ]:
import os, re
import wandb
import pandas as pd

os.makedirs('results', exist_ok=True)
api = wandb.Api()
path = f'{ENTITY+"/" if ENTITY else ""}{PROJECT}'

rows = []
for run in api.runs(path):
    if RUN_TAG not in run.name:
        continue
    m = re.search(r'layer(\d+)', run.name)
    if not m:
        continue
    s = run.summary
    rows.append({
        'model': MODEL_TAG,
        'hook_point': HOOK_POINT,
        'layer': int(m.group(1)),
        'total_loss': s.get('losses/overall_loss'),
        'mse_loss': s.get('losses/mse_loss'),
        'l1_loss': s.get('losses/l1_loss'),
        'l0': s.get('metrics/l0'),
        'explained_variance': s.get('metrics/explained_variance'),
        'dead_features': s.get('sparsity/dead_features'),
        'state': run.state,
        'passes': 3,
        'l1_coeff': RUN_TAG,
        'notes': '',
    })

df = pd.DataFrame(rows).drop_duplicates('layer').sort_values('layer').reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)
print('wrote', OUT_CSV, 'with', len(df), 'layers')
df[['layer','state','l0','explained_variance','mse_loss','dead_features']]


## 4 — Health check
Flags anything that doesn't look right so you don't eyeball 12 rows by hand.


In [ ]:
issues = []
for _, r in df.iterrows():
    lyr = int(r['layer'])
    if r['state'] != 'finished':
        issues.append(f'layer {lyr}: run state is {r["state"]} (crashed or auto-killed?)')
    if pd.notna(r['l0']) and abs(r['l0'] - 32) > 2:
        issues.append(f'layer {lyr}: L0={r["l0"]:.1f} (expected ~32)')
    if pd.notna(r['explained_variance']) and r['explained_variance'] < 0.9:
        issues.append(f'layer {lyr}: explained_variance={r["explained_variance"]:.3f} (<0.90)')

missing = sorted(set(range(12)) - set(df['layer'].astype(int)))
if missing:
    issues.append(f'missing layers (no finished run found): {missing}')

print('\n'.join(issues) if issues else 'All 12 layers look healthy: finished, L0~32, EV>=0.90.')


## 5 — Plot across layers
L0, explained variance, MSE, dead features. Overlays any other `results/sae_metrics*.csv`
(e.g. the old relu runs) so you can compare topk vs relu.


In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

frames = []
for f in sorted(glob.glob('results/sae_metrics*.csv')):
    d = pd.read_csv(f)
    d['source'] = f.split('/')[-1]
    frames.append(d)
alldf = pd.concat(frames, ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
panels = [
    ('l0', 'L0 (active features / patch)', axes[0,0]),
    ('explained_variance', 'Explained variance (higher=better)', axes[0,1]),
    ('mse_loss', 'MSE (raw; not cross-layer comparable)', axes[1,0]),
    ('dead_features', 'Dead features', axes[1,1]),
]
for col, title, ax in panels:
    if col not in alldf.columns:
        ax.set_visible(False); continue
    for (hook, tag), g in alldf.dropna(subset=[col]).groupby(['hook_point', 'l1_coeff']):
        g = g.sort_values('layer')
        ax.plot(g['layer'], g[col], marker='o', label=f'{hook} / {tag}')
    ax.set_title(title); ax.set_xlabel('layer'); ax.grid(alpha=0.3); ax.legend(fontsize=7)

axes[0,0].axhspan(10, 60, color='green', alpha=0.08)
plt.tight_layout()
plt.savefig('results/sae_eval.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved results/sae_eval.png')
